In [3]:
import re
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

def _limpar_texto(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r'[^a-zA-Z0-9áéíóúãõâêôçÁÉÍÓÚÃÕÂÊÔÇ\s]', ' ', s)
    s = re.sub(r'\b\d+\b', ' ', s)           # remove tokens puramente numéricos
    return re.sub(r'\s+', ' ', s).strip()


df = pd.read_csv("../datasets/phishing.csv", sep=",", decimal=".")

if "Email Text" not in df.columns:
    bow_cols = [c for c in df.columns if c.startswith("bow_")]
    if not bow_cols:
        raise ValueError(
            "Coluna 'Email Text' não existe e não achei colunas 'bow_*'. "
            "Use o phishing.csv original."
        )

# cria alvo se necessário
if "Email Type_Phishing Email" not in df.columns:
    if "Email Type" in df.columns:
        df["Email Type_Phishing Email"] = df["Email Type"].astype(str).str.lower().str.contains("phishing").astype(int)
    else:
        df["Email Type_Phishing Email"] = 0  # fallback

# se tiver texto, gerar BoW
if "Email Text" in df.columns:
    corpus = df["Email Text"].astype(str).apply(_limpar_texto)
    if corpus.dropna().str.strip().eq("").all():
        raise ValueError("Todos os textos ficaram vazios após a limpeza.")

    vectorizer = CountVectorizer(
        lowercase=True,
        stop_words=None,
        max_features=2500,
        min_df=5,
        max_df=0.8,
    )
    X = vectorizer.fit_transform(corpus)
    print("BoW shape:", X.shape)

    bow_cols = [f"bow_{t}" for t in vectorizer.get_feature_names_out()]
    bow_df = pd.DataFrame.sparse.from_spmatrix(X, columns=bow_cols)

    base = df.drop(columns=["Email Text"], errors="ignore")
    out_df = pd.concat([base.reset_index(drop=True), bow_df.reset_index(drop=True)], axis=1)
else:
    # já transformado com bow_*
    out_df = df.copy()

out_df.to_csv("../datasets/phishing_transformado.csv", index=False, sep=";", decimal=".")

BoW shape: (18650, 2500)
